# Vertex AI Platform Core — Hands-On Lab
### GCP Training Program — Day 4, Module 5: Vertex AI Platform Core

**What this notebook covers:** Vertex AI's architecture, a live custom training job, a small
hyperparameter tuning job, Feature Store, and Experiments/Metadata tracking — plus a guided (not
live) walkthrough of AutoML and Managed Notebooks/Colab Enterprise.

## ⚠️ Cost & trial-account notes — read this first
This day is a different cost profile than Days 1-2. Storage and Pub/Sub billed by the byte/message;
**Vertex AI training and serving bill by the node-hour**, so an idle deployed endpoint or a
forgotten training job costs money every minute it's alive, not just when it's actively used.
To keep this safe on a GCP free-trial account:
- Every resource created here uses the **smallest available machine type** (`n1-standard-4`, no
  GPU, minimal node counts) and the **smallest viable data/trial sizes**.
- **AutoML training is NOT run live in this notebook** — it routinely takes 1-3+ hours and costs
  well beyond trial-account comfort for a single run. Section 7 shows the exact code you'd use,
  clearly marked "reference only, do not execute in class."
- **Run the Cleanup section (Section 9) immediately after your demo**, not at the end of the day —
  a Feature Store or an idle custom job can keep billing quietly in the background otherwise.

**This notebook is fully self-contained.** Authenticate → Configure (asks for Project ID and
region) → everything else runs against your own project.


## 1. Authenticate This Session

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import auth
    auth.authenticate_user()
    print("Authenticated in Colab. This also configures gcloud for this session.")
else:
    print("Not running in Colab — assuming gcloud Application Default Credentials are set up.")
    print("If this is your first time, run in a terminal:")
    print("  gcloud auth application-default login")

## 2. Install & Import
`google-cloud-aiplatform` is the single SDK covering nearly everything in this notebook: training,
Experiments, Feature Store, and (via the `vertexai` module) Gen AI. `scikit-learn` provides the tiny
model we'll actually train.

In [ ]:
!pip install --quiet "google-cloud-aiplatform>=1.60.0" scikit-learn

In [ ]:
import time
import os
from google.cloud import aiplatform

print("google-cloud-aiplatform imported successfully")

## 3. Configure Your Project & Region
Vertex AI is regional — pick a region close to you that supports the features used here
(`us-central1` has the broadest feature support and is a safe default if unsure).
A Cloud Storage bucket is also created as a staging area — Vertex AI training and pipelines
always need somewhere to stage code and write output artifacts.

In [ ]:
PROJECT_ID = input("Enter your GCP Project ID: ").strip()

REGION = input("Enter your Vertex AI region [default: us-central1]: ").strip()
if not REGION:
    REGION = "us-central1"

_suffix = int(time.time())
STAGING_BUCKET = f"gs://{PROJECT_ID}-vertex-staging-{_suffix}"
EXPERIMENT_NAME = f"training-experiment-{_suffix}"

os.environ["PROJECT_ID"] = PROJECT_ID
os.environ["REGION"] = REGION
os.environ["STAGING_BUCKET_NAME"] = STAGING_BUCKET.replace("gs://", "")

!gcloud config set project {PROJECT_ID}
!gsutil mb -l {REGION} {STAGING_BUCKET}

aiplatform.init(project=PROJECT_ID, location=REGION, staging_bucket=STAGING_BUCKET, experiment=EXPERIMENT_NAME)
print("Vertex AI SDK initialized. Staging bucket:", STAGING_BUCKET)

## 4. Vertex AI Architecture Overview
Before touching any single feature, it helps the class to see how the pieces fit together —
everything in this notebook is one corner of the same platform:

| Layer | What it does | Covered in |
|---|---|---|
| **Workbench / Colab Enterprise** | Managed notebook environments for development | Section 5 |
| **Training** (Custom Jobs, AutoML, HPT) | Runs training code on managed infrastructure | Sections 6-7 |
| **Feature Store** | Centralized, low-latency feature serving for training & inference | Section 8 |
| **Experiments & Metadata** | Tracks params/metrics/lineage across training runs | Section 9 |
| **Model Registry** | Versioned home for trained models (Day 5 covers this in depth) | — |
| **Prediction / Endpoints** | Serves a registered model for online inference | — |
| **Pipelines** | Orchestrates the above into a repeatable, versioned workflow (Day 5) | — |

A quick live look at what's already enabled/available in your project:

In [ ]:
!gcloud ai models list --region={REGION} --limit=5
!gcloud services list --enabled --filter="name:aiplatform.googleapis.com"

## 5. Managed Notebooks & Colab Enterprise (Guided Walkthrough)
**Why this is a walkthrough, not a live exercise:** you're already running this lab *inside* Colab —
spinning up a second managed notebook environment mid-class adds cost and confusion without
teaching anything new experientially. Walk through this conceptually instead:

- **Vertex AI Workbench** — a managed JupyterLab instance backed by a real Compute Engine VM you
  control (choose machine type, GPU, idle-shutdown policy). Bills by the hour the VM is running,
  regardless of whether you're actively using it — the #1 reason to set an idle-shutdown timeout.
- **Colab Enterprise** — Google-managed, serverless notebook runtime integrated into Vertex AI,
  no VM to manage yourself, IAM-integrated, designed for teams needing shared/governed access.
- **When to recommend which:** Workbench for teams needing full VM control (custom images, GPUs,
  persistent local state); Colab Enterprise for lighter-weight, less ops-heavy team notebooks.

If you do want to show a real Workbench instance live, the smallest possible footprint is below —
**create it right before the demo and delete it within a few minutes**, since it bills whether or
not anyone is actively typing in it.

In [ ]:
# OPTIONAL — only run if you want to show a real Workbench instance live.
# Uses the smallest supported machine type. Delete immediately after (see the matching cell below).

WORKBENCH_NAME = f"training-workbench-{_suffix}"

# !gcloud workbench instances create {WORKBENCH_NAME} \
#     --location={REGION}-a \
#     --machine-type=e2-standard-2

print("Cell intentionally commented out — uncomment only if you plan to demo a real Workbench instance live.")

In [ ]:
# Matching cleanup for the optional Workbench instance above — run this right after the demo.
# !gcloud workbench instances delete {WORKBENCH_NAME} --location={REGION}-a --quiet

print("Cell intentionally commented out — matches the optional creation cell above.")

## 6. Custom Training Job
### 6.1 The Dataset & Training Script
We use `scikit-learn`'s built-in **Iris dataset** (150 rows, 4 numeric features, 3 flower species) —
the smallest, fastest classification dataset available, so the whole training job finishes in under
a minute rather than tying up class time. The training script itself is written to a local file,
exactly as Vertex AI expects for a Custom Training Job: a self-contained Python script that reads
data, trains a model, and writes the trained model artifact to the path Vertex AI provides via the
`AIP_MODEL_DIR` environment variable.

In [ ]:
os.makedirs("training_app", exist_ok=True)

with open("training_app/train.py", "w") as f:
    f.write('''
import os
import joblib
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# Load the tiny built-in Iris dataset — no external download needed
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=50, max_depth=4, random_state=42)
model.fit(X_train, y_train)

accuracy = model.score(X_test, y_test)
print(f"Test accuracy: {accuracy:.4f}")

# Vertex AI provides this env var pointing at the GCS path to write the model to
model_dir = os.environ.get("AIP_MODEL_DIR", "./model_output/")
os.makedirs(model_dir if not model_dir.startswith("gs://") else ".", exist_ok=True)
local_path = "model.joblib"
joblib.dump(model, local_path)

if model_dir.startswith("gs://"):
    from google.cloud import storage
    bucket_name, blob_prefix = model_dir.replace("gs://", "").split("/", 1)
    client = storage.Client()
    blob = client.bucket(bucket_name).blob(blob_prefix.rstrip("/") + "/model.joblib")
    blob.upload_from_filename(local_path)
    print(f"Model uploaded to {model_dir}")
else:
    print(f"Model saved locally to {model_dir}")
''')

print("Wrote training_app/train.py")

### 6.2 Submit the Custom Training Job
**Cost/time note:** `n1-standard-4` with no GPU is the smallest, cheapest machine type Vertex AI
Custom Training reliably supports — this job trains in well under a minute of actual compute, but
there's a few minutes of container provisioning overhead on either side, which is normal.

In [ ]:
job = aiplatform.CustomTrainingJob(
    display_name=f"iris-training-{_suffix}",
    script_path="training_app/train.py",
    container_uri="us-docker.pkg.dev/vertex-ai/training/scikit-learn-cpu.1-0:latest",
    requirements=["joblib"],
)

model = job.run(
    replica_count=1,
    machine_type="n1-standard-4",
    sync=True,  # block until the job finishes so the class sees the result live
)
print("Training job complete.")

## 7. Hyperparameter Tuning (HPT)
HPT wraps the *same* training job in a search over parameter combinations. To stay fast and cheap
for a live class, we cap it at **3 trials, 1 running in parallel** — enough to demonstrate the
mechanism (multiple trials, each with different hyperparameters, tracked and compared automatically)
without the 20-50+ trial counts a real production search would use.

In [ ]:
from google.cloud.aiplatform import hyperparameter_tuning as hpt

hpt_job_source = aiplatform.CustomJob.from_local_script(
    display_name=f"iris-hpt-{_suffix}",
    script_path="training_app/train.py",
    container_uri="us-docker.pkg.dev/vertex-ai/training/scikit-learn-cpu.1-0:latest",
    requirements=["joblib"],
    machine_type="n1-standard-4",
    replica_count=1,
)

hpt_job = aiplatform.HyperparameterTuningJob(
    display_name=f"iris-hpt-job-{_suffix}",
    custom_job=hpt_job_source,
    metric_spec={"accuracy": "maximize"},
    parameter_spec={
        "max_depth": hpt.IntegerParameterSpec(min=2, max=8, scale="linear"),
    },
    max_trial_count=3,
    parallel_trial_count=1,
)

print("Note: the training script above doesn't yet parse --max_depth as a CLI arg or report a")
print("hyperparameter-tuning metric via cloudml-hypertune — that wiring is the realistic next step")
print("an engineer would add. This cell shows the HPT *job configuration* pattern; uncomment run()")
print("below once the script is wired to accept/report the tuned parameter.")

# hpt_job.run(sync=True)

## 8. AutoML (Reference Only — Do Not Run Live)
**Why this is reference-only:** AutoML training on even a small tabular dataset typically takes
1-3+ hours and consumes meaningfully more budget than a trial account is comfortable spending on a
single demo run. The code below is real, correct, and would work — walk through it with the class,
then either show results from a run you did before class, or point to the console.

In [ ]:
# REFERENCE ONLY — DO NOT UNCOMMENT AND RUN DURING A LIVE TRIAL-ACCOUNT SESSION.
# This is the real, correct pattern for an AutoML tabular training job:

# dataset = aiplatform.TabularDataset.create(
#     display_name="iris-automl-dataset",
#     gcs_source="gs://your-bucket/iris.csv",
# )
#
# automl_job = aiplatform.AutoMLTabularTrainingJob(
#     display_name="iris-automl-training",
#     optimization_prediction_type="classification",
# )
#
# automl_model = automl_job.run(
#     dataset=dataset,
#     target_column="species",
#     budget_milli_node_hours=1000,  # 1 node-hour — the practical minimum AutoML allows
#     model_display_name="iris-automl-model",
# )

print("Reference code above — intentionally not executed. Walk through it, don't run it live.")

## 9. Feature Store
**Caveat for the class:** Vertex AI Feature Store's API has evolved significantly across SDK
versions (there's a legacy Feature Store and a newer BigQuery-backed Feature Store surface) — check
your installed `google-cloud-aiplatform` version against current docs if a call here doesn't match.

### 9.1 Create a Minimal Feature Store
One online-serving node is the smallest configuration available — still bills per node-hour, so
this is a strong candidate to delete right after the demo rather than leaving it running.

In [ ]:
from google.cloud.aiplatform import Featurestore

FEATURESTORE_ID = f"training_fs_{_suffix}"

featurestore = Featurestore.create(
    featurestore_id=FEATURESTORE_ID,
    online_store_fixed_node_count=1,  # smallest possible online-serving footprint
    sync=True,
)
print("Created Feature Store:", featurestore.resource_name)

### 9.2 Create an Entity Type & Features
An **entity type** groups related features (here: a "flower" entity with the same 4 Iris
measurements used for training) — in production this is typically something like `user` or
`product`, with dozens of features kept in sync for both training and low-latency serving.

In [ ]:
entity_type = featurestore.create_entity_type(
    entity_type_id="flower",
    description="A single flower measurement, matching the Iris training features",
)

entity_type.create_feature(feature_id="sepal_length", value_type="DOUBLE")
entity_type.create_feature(feature_id="sepal_width", value_type="DOUBLE")
entity_type.create_feature(feature_id="petal_length", value_type="DOUBLE")
entity_type.create_feature(feature_id="petal_width", value_type="DOUBLE")

print("Created entity type 'flower' with 4 features.")

### 9.3 Ingest a Few Sample Feature Values & Read Them Back
A tiny in-memory `pandas` DataFrame stands in for a real feature pipeline — the point here is the
ingest-then-serve round trip, not data volume.

In [ ]:
import pandas as pd

sample_features = pd.DataFrame({
    "flower": ["flower-001", "flower-002", "flower-003"],
    "sepal_length": [5.1, 6.2, 4.9],
    "sepal_width": [3.5, 2.9, 3.0],
    "petal_length": [1.4, 4.3, 1.4],
    "petal_width": [0.2, 1.3, 0.2],
    "update_time": pd.to_datetime(["2024-01-01", "2024-01-01", "2024-01-01"]),
})

entity_type.ingest_from_df(
    feature_ids=["sepal_length", "sepal_width", "petal_length", "petal_width"],
    feature_time="update_time",
    df_source=sample_features,
    entity_id_field="flower",
)

online_features = entity_type.read(entity_ids=["flower-001", "flower-002"])
print("Read back from online serving:")
print(online_features)

## 10. Experiments & Metadata Tracking
Vertex AI Experiments tracks parameters, metrics, and lineage across training runs — the point is
comparing multiple runs later, not just logging one. We log the Section 6 training run's
hyperparameters and resulting accuracy as a tracked run.

In [ ]:
with aiplatform.start_run(f"iris-run-{_suffix}") as run:
    aiplatform.log_params({
        "n_estimators": 50,
        "max_depth": 4,
        "model_type": "RandomForestClassifier",
    })
    aiplatform.log_metrics({
        "test_accuracy": 0.9667,  # from the training job output in Section 6.2 — copy the real value printed there
    })

print("Logged a run to Experiment:", EXPERIMENT_NAME)

In [ ]:
experiment_df = aiplatform.get_experiment_df(EXPERIMENT_NAME)
print(experiment_df)

## 11. Cleanup
Run this **immediately** after your demo, not at the end of the day — the Feature Store's online
node and any Workbench instance from Section 5 bill continuously while they exist.

In [ ]:
# Feature Store (deletes entity types and features along with it)
featurestore.delete(sync=True, force=True)
print("Deleted Feature Store.")

# Custom training job resources
job.delete()
print("Deleted custom training job.")

# Staging bucket
!gsutil -m rm -r {STAGING_BUCKET}
print(f"Deleted staging bucket {STAGING_BUCKET}.")

print("\nReminder: if you created a real Workbench instance in Section 5, delete it too:")
print(f"  gcloud workbench instances delete {WORKBENCH_NAME} --location={REGION}-a --quiet")